# Smart Loan Recovery System

ML-powered borrower risk segmentation, recovery prediction, and strategy optimization.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.loan_recovery import (
    LoanDataLoader, FeatureEngineer, LoanEDA, BorrowerSegmenter,
    ModelTrainer, ModelEvaluator, RecoveryStrategy, ModelExplainer,
    DashboardBuilder, ensure_dirs,
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ensure_dirs()
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading

In [ ]:
loader = LoanDataLoader()
df = loader.load()
print(df.shape)
print(df.info())
loader.target_distribution().to_frame('count')

## 2. EDA

In [ ]:
eda = LoanEDA(df)
eda.target_distribution()
plt.show()

In [ ]:
eda.correlation_heatmap()
plt.show()

In [ ]:
eda.numeric_distributions()
plt.show()

In [ ]:
print(eda.summary_report())

## 3. Feature Engineering

In [ ]:
fe = FeatureEngineer()
X_train, X_test, y_train, y_test = fe.fit_transform(df)
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
list(X_train.columns)

## 4. Borrower Segmentation

In [ ]:
seg = BorrowerSegmenter()
seg_df = seg.fit_predict(pd.concat([X_train, X_test], axis=0))
pd.DataFrame(seg.scores_).T

In [ ]:
seg.plot_elbow()
plt.show()
seg.plot_clusters_pca(seg_df)
plt.show()
seg.segment_summary(seg_df)

## 5. Model Training

In [ ]:
trainer = ModelTrainer()
trainer.train_baselines(X_train, y_train, X_test, y_test)
trainer.train_xgboost(X_train, y_train, X_test, y_test, tune=True)
trainer.comparison_df()

In [ ]:
best_model, best_name = trainer.get_best_model()
print(f'Best model: {best_name}')
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

## 6. Evaluation

In [ ]:
evaluator = ModelEvaluator()
evaluator.plot_confusion_matrix(y_test, y_pred)
plt.show()
evaluator.plot_roc_curves(y_test, y_prob)
plt.show()
evaluator.plot_comparison(trainer.comparison_df())
plt.show()

## 7. Recovery Strategy

In [ ]:
strategy = RecoveryStrategy(model=best_model, feature_engineer=fe)
result = strategy.assign_strategies(df)
result[['risk_tier', 'recommended_action', 'priority']].head(10)

In [ ]:
scenarios = strategy.scenario_analysis(df)
pd.DataFrame(scenarios).T

In [ ]:
strategy.get_borrower_breakdown(result).head(10)

## 8. Explainability (SHAP)

In [ ]:
explainer = ModelExplainer(best_model, list(X_train.columns))
explainer.feature_importance()

In [ ]:
explainer.shap_summary(X_test)

In [ ]:
explainer.shap_waterfall(X_test, idx=0)

## 9. Dashboard (Plotly)

In [ ]:
db = DashboardBuilder()
fig = db.target_distribution(df)
fig.show()

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
fig = db.confusion_matrix_plot(cm)
fig.show()

---
*End-to-end pipeline complete. Run `streamlit run app.py` for interactive dashboard.*